# Práctica evaluable S5–S7 (Clustering y clasificación)

## Dos problemas, dos objetivos

En esta práctica trabajarás con dos problemas. En el caso no supervisado, explorarás datos sintéticos y compararás **K-Means** con el **clustering jerárquico aglomerativo** (enlaces Ward y, como ampliación, completo). En el caso supervisado, compararás cinco clasificadores de imágenes de dígitos: **KNN, Naive Bayes gaussiano, árbol de decisión, random forest y SVM**.

> **Importante:** en clustering, las etiquetas verdaderas se reservan para una comprobación externa después de seleccionar la partición con evidencia interna. En clasificación, usarás ajuste y validación para comparar los modelos; el conjunto test queda reservado para la evaluación final.

## Cómo trabajar, entregar y consultar los materiales

1. Ejecuta las celdas en orden. Completa los bloques `TODO` y no borres las preguntas.
2. Escribe las interpretaciones en las celdas **RESPUESTA**; no se solicita un informe adicional.
3. Justifica con resultados visibles: tamaños, métricas, recuentos o ejemplos concretos, e indica al menos una limitación.
4. **Clustering:** no uses las etiquetas conocidas para escoger el método ni el número de grupos. Se consultan solo al final como comparación externa.
5. **Clasificación:** ajusta y compara usando train/validación; el test se reserva para la evaluación final.
6. Conserva las líneas `YA PREPARADO — NO MODIFICAR`; completa únicamente los apartados indicados.

### Mapa de consulta: ejemplos de código en las sesiones

Las referencias indican secciones de los notebooks de sesión. Busca allí ejemplos ejecutables de las operaciones; adapta los nombres de variables y mantén el protocolo de esta práctica.

- **S5, clustering:** [estandarización y efecto de la escala (5.1)](../sesion5/s5.ipynb); [distancias, enlaces, dendrogramas y cortes jerárquicos (5.2)](../sesion5/s5.ipynb); [WSS e índices internos, incluidos Silhouette, Calinski–Harabasz, Davies–Bouldin y Dunn (5.2.5–5.2.7)](../sesion5/s5.ipynb); [K-Means, WSS y elección de $k$ (5.5)](../sesion5/s5.ipynb).
- **S6, clasificación:** [KNN: vecinos, escalas, partición, predicción y matriz de confusión (6.1)](../sesion6/s6.ipynb); [Naive Bayes: ajuste, probabilidades posteriores y evaluación por clase (6.2)](../sesion6/s6.ipynb).
- **S7, clasificación:** [árboles y poda (7.1)](../sesion7/s7.ipynb); [random forest y evaluación OOB (7.2)](../sesion7/s7.ipynb); [SVM, escalado con pipeline y búsqueda de hiperparámetros (7.3)](../sesion7/s7.ipynb).
- **S1–S2:** descriptivas, gráficos y lectura de distribuciones.

La selección de clustering combina dendrograma o curva de WSS, índices internos, tamaños y lectura de los grupos. Ninguna medida aislada determina la respuesta.

### Pasos guiados de esta práctica

La plantilla deja fijadas la semilla, los datasets, las particiones supervisadas y las funciones de evaluación. Así puedes centrarte en completar las operaciones, revisar sus salidas y explicar las decisiones.

En clustering, no hay train/test convencional: el método resume la estructura de las entradas disponibles. La etiqueta de `make_blobs` se mantiene aparte y sirve solo para calcular una comparación externa después de seleccionar la partición con evidencia interna.

En clasificación, las particiones sí tienen un papel distinto: train sirve para ajustar y hacer la validación cruzada; validación externa ayuda a comparar familias; test se reserva para el informe final. No uses test para escoger hiperparámetros ni para elegir qué clases analizar.

En cada apartado sigue el ciclo: revisa la pregunta, ejecuta las líneas preparadas, completa los `TODO`, guarda los objetos necesarios y escribe una respuesta basada en las salidas.

Las instrucciones de cada apartado indican qué resultado hay que obtener y dónde escribir su interpretación.

## 1. Preparar el entorno común y la semilla

Ejecuta en orden las celdas preparadas de esta sección. Importan las herramientas y definen las funciones para generar y explorar datos, agrupar observaciones y comparar clasificadores. La semilla fija las operaciones aleatorias para facilitar la repetición.

Las métricas multiclase se resumen con exactitud y promedios macro. El promedio macro calcula cada clase por separado y luego les da el mismo peso, aunque tengan distinto número de ejemplos.

### Dónde encontrar las funciones preparadas

Todas están definidas en las celdas de esta sección. Para localizarlas rápidamente, busca su nombre en el notebook:

- **Primera celda de código**, junto a las importaciones: `evaluar_particion` y `dunn_index` calculan medidas internas; `dibujar_dendrograma` representa la jerarquía completa y `ari_externo` calcula el acuerdo externo. También están `metricas_multiclase`, `crear_busqueda_clasificador` y `copiar_mejor_estimador` para clasificación.
- **Celda siguiente:** `matriz_confusion_multiclase` construye la matriz de confusión.
- **Última celda de código de esta sección:** `ajustar_kmeans` y `ajustar_jerarquico` ajustan clustering; `dibujar_dendrograma_resumido` muestra un resumen legible de las últimas fusiones.

In [ ]:
# YA PREPARADO — NO MODIFICAR: bibliotecas y semilla para clustering y clasificación.
import matplotlib.pyplot as plt  # Crea gráficos para explorar observaciones y resultados.
import numpy as np  # Proporciona arreglos y operaciones numéricas.
import pandas as pd  # Organiza los datos y resultados en tablas.
from scipy.cluster.hierarchy import dendrogram, linkage  # Calcula y dibuja jerarquías aglomerativas.
from scipy.spatial.distance import cdist, pdist  # Calcula distancias entre puntos y grupos.
from sklearn.base import clone  # Copia una configuración seleccionada para el ajuste final.
from sklearn.cluster import AgglomerativeClustering, KMeans  # Métodos de clustering evaluados en S5.
from sklearn.datasets import load_digits, make_blobs  # Datos de dígitos y generador sintético de grupos.
from sklearn.ensemble import RandomForestClassifier  # Bosque de árboles de clasificación.
from sklearn.metrics import (
    accuracy_score,  # Fracción de clasificaciones correctas.
    adjusted_rand_score,  # Compara al final los grupos con las etiquetas externas.
    calinski_harabasz_score, davies_bouldin_score, silhouette_score,  # Índices internos de clustering.
    confusion_matrix,  # Cuenta combinaciones entre clase real y predicha.
    f1_score, precision_score, recall_score,  # Métricas que resumen rendimiento por clase.
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split  # Búsqueda y particiones estratificadas.
from sklearn.naive_bayes import GaussianNB  # Clasificador Naive Bayes gaussiano.
from sklearn.neighbors import KNeighborsClassifier  # Clasificador de vecinos más próximos.
from sklearn.pipeline import Pipeline  # Encadena escalado y clasificador sin fuga de información.
from sklearn.preprocessing import StandardScaler  # Estandariza variables usando solo datos de ajuste.
from sklearn.svm import SVC  # Clasificador de máquinas de vectores soporte.
from sklearn.tree import DecisionTreeClassifier  # Árbol de decisión para clasificación.

SEMILLA = 42  # YA PREPARADO: valor fijo para poder repetir muestras y particiones.
np.random.seed(SEMILLA)  # Inicializa el generador global de NumPy.
plt.rcParams.update({"figure.figsize": (8, 4), "figure.dpi": 100})  # Ajusta el tamaño inicial de los gráficos.
pd.set_option("display.max_columns", 30)  # Permite mostrar varias columnas de las tablas.


def metricas_multiclase(reales, predichas):  # Resume el rendimiento de etiquetas de varias clases.
    return pd.Series({  # Devuelve medidas con nombres fáciles de interpretar.
        "Exactitud": accuracy_score(reales, predichas),  # Proporción total de etiquetas correctas.
        "Sensibilidad macro": recall_score(reales, predichas, average="macro", zero_division=0),  # Promedia recuperación por clase.
        "Precisión macro": precision_score(reales, predichas, average="macro", zero_division=0),  # Promedia precisión por clase.
        "F1 macro": f1_score(reales, predichas, average="macro", zero_division=0),  # Promedia F1 con igual peso por clase.
    })  # Termina la serie de métricas.


def dunn_index(X, etiquetas):  # Compara separación mínima entre grupos con diámetro máximo interno.
    valores = np.asarray(X)  # Convierte las características a matriz numérica.
    grupos = np.unique(etiquetas)  # Obtiene los identificadores distintos de grupo.
    diametros = []  # Guardará el mayor par de distancias dentro de cada grupo.
    separaciones = []  # Guardará distancias mínimas entre parejas de grupos.
    for grupo in grupos:  # Recorre cada grupo para medir su extensión.
        puntos = valores[np.asarray(etiquetas) == grupo]  # Selecciona las observaciones asignadas.
        diametros.append(pdist(puntos).max() if len(puntos) > 1 else 0.0)  # Diámetro máximo dentro del grupo.
    for i, grupo_a in enumerate(grupos):  # Recorre los grupos para comparar sus separaciones.
        for grupo_b in grupos[i + 1:]:  # Evita comparar un grupo consigo mismo y no duplica parejas.
            puntos_a = valores[np.asarray(etiquetas) == grupo_a]  # Selecciona puntos del primer grupo.
            puntos_b = valores[np.asarray(etiquetas) == grupo_b]  # Selecciona puntos del segundo grupo.
            separaciones.append(cdist(puntos_a, puntos_b).min())  # Guarda la distancia mínima entre esos grupos.
    mayor_diametro = max(diametros)  # Encuentra el grupo más extendido.
    return min(separaciones) / mayor_diametro if mayor_diametro > 0 else np.nan  # Dunn alto indica grupos compactos y separados.


def evaluar_particion(X, etiquetas):  # Calcula medidas internas y WSS para comparar particiones.
    valores = np.asarray(X)  # Convierte las características a matriz numérica.
    grupos = np.asarray(etiquetas)  # Convierte etiquetas de grupo a array.
    wss = sum(((valores[grupos == g] - valores[grupos == g].mean(axis=0)) ** 2).sum() for g in np.unique(grupos))  # Suma cuadrados dentro de grupos.
    return pd.Series({  # Reúne métricas en una serie comparable.
        "WSS": wss,  # Menor es mejor, pero siempre disminuye al aumentar k.
        "Silhouette": silhouette_score(valores, grupos),  # Mayor es mejor; combina cohesión y separación.
        "Calinski-Harabasz": calinski_harabasz_score(valores, grupos),  # Mayor suele indicar partición más separada.
        "Davies-Bouldin": davies_bouldin_score(valores, grupos),  # Menor suele indicar grupos más compactos y separados.
        "Dunn": dunn_index(valores, grupos),  # Mayor indica separación frente al diámetro de los grupos.
    })  # Devuelve todas las métricas internas.


def ari_externo(referencia, grupos):  # Compara particiones solo después de cerrar la selección.
    return adjusted_rand_score(referencia, grupos)  # Devuelve el acuerdo ajustado por azar.


def dibujar_dendrograma(X, metodo="ward"):  # Representa fusiones jerárquicas en forma de dendrograma.
    Z = linkage(X, method=metodo)  # Calcula la matriz de enlaces para el método elegido.
    dendrogram(Z)  # Dibuja las fusiones y sus distancias.
    plt.title(f"Dendrograma · enlace {metodo}")  # Nombra el método utilizado.
    plt.xlabel("Observaciones")  # Etiqueta el eje de observaciones.
    plt.ylabel("Distancia de fusión")  # Etiqueta la altura a la que se unen grupos.
    plt.show()  # Presenta el dendrograma.


CV_CLASIFICACION = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEMILLA)  # Pliegues reproducibles y estratificados.


def crear_busqueda_clasificador(estimador, rejilla):  # Crea la búsqueda común para ajustar cada clasificador.
    return GridSearchCV(  # Devuelve búsqueda de hiperparámetros con validación cruzada interna.
        estimador, rejilla, scoring="f1_macro", cv=CV_CLASIFICACION, refit=True, n_jobs=1
    )  # Selecciona por F1 macro y reajusta el mejor candidato con el entrenamiento completo.


def copiar_mejor_estimador(busqueda):  # Prepara una copia del mejor pipeline encontrado con validación cruzada.
    return clone(busqueda.best_estimator_)  # Permite reajustar la configuración seleccionada con train + validación.


def reunir_series_metricas(diccionario_de_series):  # Convierte series nombradas en filas de una tabla.
    return pd.concat(diccionario_de_series, axis=1).T  # Alinea las métricas por nombre y transpone candidatos a filas.

In [ ]:
def matriz_confusion_multiclase(reales, predichas, clases):  # Devuelve filas predichas y columnas reales.
    return confusion_matrix(reales, predichas, labels=clases).T  # Transpone la convención de sklearn para esta práctica.

In [ ]:
def ajustar_kmeans(X, k, semilla=SEMILLA):  # Ajusta K-Means y devuelve el modelo y las etiquetas de grupo.
    modelo = KMeans(n_clusters=k, n_init=20, random_state=semilla)  # Usa varios inicios para reducir dependencia de una inicialización.
    etiquetas = modelo.fit_predict(X)  # Aprende centroides y asigna cada observación a un grupo.
    return modelo, etiquetas  # Devuelve el objeto para leer inertia_ y las etiquetas para evaluar.


def ajustar_jerarquico(X, k, enlace="ward"):  # Ajusta un clustering aglomerativo y devuelve sus etiquetas.
    modelo = AgglomerativeClustering(n_clusters=k, linkage=enlace)  # Define número de grupos y método de enlace.
    etiquetas = modelo.fit_predict(X)  # Construye los grupos y asigna una etiqueta a cada observación.
    return modelo, etiquetas  # Devuelve el modelo y la partición resultante.


def dibujar_dendrograma_resumido(X, metodo="ward", ultimas_fusiones=25):  # Facilita la lectura de jerarquías grandes.
    Z = linkage(X, method=metodo)  # Calcula las fusiones con el enlace indicado.
    plt.figure(figsize=(12, 5))  # Mantiene un tamaño legible para ambos enlaces.
    dendrogram(Z, truncate_mode="lastp", p=ultimas_fusiones, show_leaf_counts=True)  # Resume las últimas fusiones y cuenta observaciones agrupadas.
    plt.title(f"Dendrograma resumido · enlace {metodo}")  # Identifica el enlace representado.
    plt.xlabel("Grupos o subárboles resumidos")  # Las hojas pueden representar varias observaciones.
    plt.ylabel("Distancia de fusión")  # Indica la altura a la que se fusionan.
    plt.show()  # Presenta el dendrograma.

## 2. Cargar y auditar los datasets

La práctica utiliza dos propuestas sin descargas externas:

| Caso | Datos | Objetivo |
|---|---|---|
| Clustering | `make_blobs`, 1 000 puntos con dos características numéricas; se generan tres etiquetas de origen | Comparar K-Means y cortes jerárquicos usando evidencia interna; las etiquetas se reservan para la comparación externa final |
| Clasificación | Dígitos escritos a mano (`load_digits`), 1 797 imágenes de 8 × 8 píxeles | Predecir una de diez cifras a partir de 64 intensidades de píxel |

**Importante:** las tres etiquetas de origen describen cómo se generaron los datos; no son una respuesta que haya que reproducir ni fijan el valor esperado de $k$. El método y el número de grupos se eligen con evidencia interna; el ARI permite comparar esa elección con las etiquetas solo después.

### 2.0 Preparar los datos

Ejecuta la celda preparada para generar de forma reproducible los datos sintéticos y cargar las imágenes de dígitos. La generación deforma suavemente las nubes y añade curvatura a una de ellas; además, una coordenada se expresa en una escala diez veces mayor para estudiar cómo afecta la escala a las distancias. Identifica qué objetos contienen las características y cuál contiene la respuesta supervisada.

Aunque se generan tres etiquetas de origen, esto describe el mecanismo de generación, no el número de grupos que debas escoger. Mantén `etiquetas_externas` reservadas: no las uses para formar ni elegir grupos. La selección se justifica con evidencia interna; después, el ARI sirve para comparar tu partición con esas etiquetas.

In [ ]:
# YA PREPARADO — NO MODIFICAR: genera datos sintéticos reproducibles y carga dígitos.
# Los centros y dispersiones dejan solapamiento moderado; la curva sigue siendo una deformación visible.
X_grupos_array, etiquetas_externas = make_blobs(
    n_samples=1000,
    centers=[[-4.0, -1.6], [0.0, 5.0], [5.0, -2.3]],
    cluster_std=[2.2, 1.8, 2.6],
    random_state=SEMILLA,
)  # Tres nubes con solapamiento moderado.
transformacion = np.array([[1.0, 0.25], [0.30, 1.0]])  # Alarga e inclina suavemente las nubes.
X_grupos_array = X_grupos_array @ transformacion

# Durante la generación, curva ligeramente la nube de origen 1 sin cambiar el orden de las filas.
mascara_curva = etiquetas_externas == 1
coordenadas_curva = X_grupos_array[mascara_curva, 0]
centro_curva = coordenadas_curva.mean()
X_grupos_array[mascara_curva, 1] += 0.10 * (coordenadas_curva - centro_curva) ** 2

X_sintetico = pd.DataFrame(X_grupos_array, columns=["característica_1", "característica_2"])
X_sintetico["característica_1"] *= 10  # Cambia la escala de una coordenada para estudiar su efecto en las distancias.

digitos = load_digits(as_frame=True)  # Carga imágenes pequeñas incluidas con scikit-learn.
X_digitos = digitos.data.copy()  # Cada fila contiene 64 intensidades de píxel.
y_digitos = digitos.target.copy().astype(int)  # La etiqueta es una cifra entera de 0 a 9.

print("Datos sintéticos:", X_sintetico.shape, "etiquetas externas reservadas:", etiquetas_externas.shape)
print("Dígitos:", X_digitos.shape, "clases:", sorted(y_digitos.unique()))

### 2.1. Revisar estructura y calidad

Una tabla puede tener filas repetidas, valores ausentes o tipos no esperados. La siguiente inspección muestra dimensiones, tres ejemplos, tipos y recuentos generales de calidad. Describe las variables de entrada antes de ajustar cualquier método.

**Completa los `TODO`** con resúmenes numéricos. En los datos de dígitos, cada valor es una intensidad de píxel; la etiqueta se guarda en `y_digitos`, no forma parte de `X_digitos`.

### Pasos y pistas para revisar estructura y calidad

1. Comprueba las dimensiones y muestra unas pocas filas de cada tabla.
2. Revisa los tipos con `dtypes`.
3. Cuenta valores ausentes con `isna().sum()` y filas duplicadas completas con `duplicated().sum()`.

Haz las comprobaciones por separado para cada tabla. No elimines registros solo porque aparezcan duplicados: primero valora si son observaciones repetidas o casos distintos con valores iguales. Que dos filas tengan un píxel o una coordenada iguales no las hace duplicadas.

En la respuesta, separa hechos observados de interpretaciones. Si no encuentras ausentes, eso no prueba que cada píxel esté correctamente medido; solo indica que no hay celdas vacías.

In [ ]:
# TODO: Informa las dimensiones de X_sintetico y X_digitos.
# Pista de código: consulta el atributo .shape; devuelve (filas, columnas).

# TODO: Muestra las tres primeras filas de cada conjunto.
# Pista de código: el método .head(3) muestra las primeras tres observaciones.

In [ ]:
# TODO: Resume los tipos de datos y cuenta cuántas columnas hay de cada tipo en ambas tablas.
# Pista de código: revisa .dtypes y agrupa el resultado con .value_counts().

# TODO: Cuenta los valores ausentes por tabla y el total de filas duplicadas completas.
# Pista de código: para ausentes, encadena .isna().sum(); para duplicados, usa .duplicated().sum().
# Pista: aplica las comprobaciones a X_sintetico y X_digitos por separado; no elimines ni modifiques filas.

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 2.1</strong><br><br><b>1.</b> ¿Cuántas filas y columnas tiene cada tabla? ¿Qué tipos de datos aparecen y qué observas en las primeras filas?<br><br><b>2.</b> ¿Cuántos valores ausentes y filas duplicadas hay en cada tabla? ¿Qué problemas ayudan a detectar estas comprobaciones y cuáles no permiten descartar?<br><br><em>Pista: distingue la ausencia de celdas vacías de la validez o exactitud de los valores; una fila duplicada tampoco implica automáticamente que deba eliminarse.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br><br>Tu respuesta aquí. Resume dimensiones, tipos, valores ausentes y duplicados de cada tabla. Explica qué permiten concluir esas comprobaciones y menciona una limitación que no detectan.</div>

### 2.2. Analizar los datos sintéticos

Examina los rangos y la dispersión de las dos coordenadas de `X_sintetico` antes de estudiar los grupos. Resume cada característica por separado y utiliza las estadísticas observadas para responder la pregunta, sin consultar las etiquetas externas.

### Pistas para describir las coordenadas

En el resumen, lee una característica por fila: `min` y `max` indican el rango observado, mientras que `mean` y `std` describen su centro y dispersión. Compara las coordenadas entre sí, considerando que representan posiciones en el espacio sintético.

Las etiquetas generadas por `make_blobs` se reservan para la comparación externa posterior; no se usan para esta descripción ni para elegir la partición.

In [ ]:
# TODO: resume las dos características de X_sintetico y compara sus rangos y dispersiones.
# Pista de código: describe().T muestra una característica por fila y las estadísticas en columnas.
# Para seleccionar columnas del resumen, usa .loc[:, ["min", "mean", "std", "max"]].
# Describe las diferencias en la respuesta 2.2, sin utilizar etiquetas_externas.

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 2.2</strong><br><br><b>1.</b> ¿Cuáles son los mínimos, las medias y los máximos observados de cada coordenada? ¿En qué se diferencian sus rangos?<br><br><b>2.</b> ¿Qué indican las desviaciones típicas sobre la dispersión de cada característica? ¿Por qué conviene revisar la escala antes de calcular distancias?<br><br><em>Pista: cita valores del resumen y distingue las coordenadas de las etiquetas externas.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br><br>Tu respuesta aquí. Compara los rangos y dispersiones de las dos coordenadas y explica por qué importa la escala al calcular distancias.</div>

### 2.3. Analizar las imágenes de dígitos

Resume las intensidades de los píxeles de `X_digitos` y cuenta los ejemplos de cada cifra en `y_digitos`. Examina los rangos por píxel y el balance de las clases para describir las entradas y preparar las particiones supervisadas; no utilices estos resúmenes para elegir un clasificador.

In [ ]:
# TODO: resume las intensidades de los píxeles de X_digitos por característica.
# Pista de código: describe().T sitúa los píxeles en filas y las estadísticas en columnas.
# Selecciona las columnas con .loc[:, ["min", "mean", "max"]] y revisa cómo varían entre píxeles.

In [ ]:
# TODO: Cuenta los ejemplos de cada dígito y presenta las clases en orden del 0 al 9.
# Pista de código: y_digitos.value_counts() cuenta las etiquetas; añade .sort_index() para ordenarlas por dígito.


<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 2.3 — Dígitos</strong><br><br><b>1.</b> ¿Qué rangos de intensidad observas en los píxeles? ¿Todos presentan los mismos valores mínimos, medios y máximos?<br><br><b>2.</b> ¿Las frecuencias de los diez dígitos son parecidas? ¿Por qué conviene conservar sus proporciones al crear particiones estratificadas?<br><br><em>Pista: cita estadísticas de algunos píxeles y los recuentos por clase; distingue las entradas de la etiqueta que se quiere predecir.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br><br>Tu respuesta aquí. Describe los rangos de intensidad y el balance de las cifras; explica por qué conviene estratificar las particiones.</div>

## 3. EDA y estandarización para clustering

En esta sección explorarás las dos características numéricas de los datos sintéticos antes de comparar métodos de clustering. Revisarás primero sus escalas y distribuciones, y después observarás los mismos puntos tras estandarizarlos.

La nube (el blob) se generó alrededor de varios centros con dispersiones distintas. Las etiquetas externas deben permanecer apartadas: no las uses para colorear los gráficos ni para describir o decidir los grupos. La estandarización centra y escala cada característica para que una escala numérica mayor no domine por sí sola las distancias. Conserva la misma matriz estandarizada al comparar K-Means y los métodos jerárquicos.

**Completa la exploración con estos pasos:**

1. En el apartado 3.1, resume las características originales y estandarizadas; compara medias, desviaciones y rangos.
2. En el apartado 3.2, revisa el diagrama de dispersión preparado y compara los ejes antes y después de escalar, sin usar las etiquetas externas.
3. Completa en el apartado 3.2 los histogramas o diagramas de caja y el cálculo de correlación entre las entradas.
4. Describe los patrones observados e indica qué conclusiones permiten estos resúmenes y gráficos, y cuáles no.

**Dónde mirar:** [S5, resumen y gráficos de pares (5.1.1)](../sesion5/s5.ipynb) y [S5, por qué importa estandarizar (5.1.2)](../sesion5/s5.ipynb). Para histogramas, diagramas de caja y correlación, consulta también [S1, gráficos descriptivos](../sesion1/s1.ipynb). Una correlación lineal pequeña no descarta estructura en los datos.

### 3.1. Resumir antes y después de estandarizar

En esta parte céntrate en los resúmenes numéricos.


Comprueba que las medias estandarizadas quedan aproximadamente en los valores esperados. Los rangos también cambian al escalar; comenta ese cambio sin interpretar todavía grupos ni usar las etiquetas externas.

In [ ]:
# YA PREPARADO — NO MODIFICAR: el escalador recibe solo características, nunca etiquetas.
escalador_grupos = StandardScaler()
X_sintetico_std = escalador_grupos.fit_transform(X_sintetico)

# TODO: resume las variables antes y después de estandarizar, con una variable por fila.
# Pista de código: usa describe().T para X_sintetico.
# Para el array estandarizado, pd.DataFrame(X_sintetico_std, columns=X_sintetico.columns) le asigna los mismos nombres de columna;
# después puedes resumir ese DataFrame con describe().T.



In [ ]:
# TODO: compara las medias y desviaciones típicas originales con las estandarizadas.
# Pista de código: puedes consultar .mean() y .std() en cada conjunto; las medias estandarizadas deberían quedar cerca de cero 
# y las desviaciones, cerca de uno.



<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 3.1</strong><br><br><b>1.</b> ¿Cómo cambian la media y la desviación típica de cada característica después de estandarizar? ¿Quedan aproximadamente en los valores esperados?<br><br><b>2.</b> ¿Qué ocurre con los rangos y con la característica más dispersa en escala original? Explica el cambio de unidades.<br><br><b>3.</b> ¿Por qué el escalador se ajusta solo con las características y no utiliza las etiquetas externas?<br><br><em>Pista: cita estadísticas de ambas representaciones y distingue el cambio de escala de un cambio en los datos o en sus etiquetas.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br><br>Tu respuesta aquí. Compara medias y desviaciones antes y después; cita qué pasa con los rangos y cuál característica era más dispersa. Explica el cambio de unidades y por qué las etiquetas externas no intervienen.</div>

### 3.2. Explorar geometría y distribuciones

Antes de agrupar, revisa cómo se distribuyen las dos características y si tienen rangos muy distintos. Los gráficos deben mostrar solo la geometría de los puntos; **no uses el color de las clases guardadas**. Compara los paneles antes y después de estandarizar.

In [ ]:
# Dibuja las dos variables de entrada en sus escalas originales y estandarizadas.
fig, ejes = plt.subplots(1, 2, figsize=(10, 4))  # Crea dos paneles para comparar las representaciones.
ejes[0].scatter(X_sintetico.iloc[:, 0], X_sintetico.iloc[:, 1], alpha=0.6)  # Muestra los datos originales sin etiquetas externas.
ejes[0].set(title="Escala original", xlabel="Característica 1", ylabel="Característica 2")  # Añade títulos y nombres de ejes.
ejes[1].scatter(X_sintetico_std[:, 0], X_sintetico_std[:, 1], alpha=0.6)  # Muestra los mismos puntos tras estandarizar.
ejes[1].set(title="Escala estandarizada", xlabel="Característica 1", ylabel="Característica 2")  # Permite comparar el cambio de escala.
fig.tight_layout()  # Ajusta los márgenes de los dos paneles.
plt.show()  # Presenta ambas vistas.

In [ ]:
# TODO: añade histogramas o boxplots por característica y describe su forma.
# Pista: con plt.subplots(1, 2), crea un panel para cada columna de X_sintetico y X_sintetico_std.
# En cada panel, dibuja el histograma de esa columna con ax.hist(...), o el boxplot con ax.boxplot(...).
# Añade título y nombre de la característica; compara centro, dispersión, asimetría y posibles valores alejados.

In [ ]:
# TODO: calcula correlación entre las dos variables y explica qué mide aquí.
# Pista de código: selecciona las dos columnas de X_sintetico y X_sintetico_std y usa el método .corr().
# La correlación de Pearson resume la dirección y fuerza de una relación lineal; interpreta el signo y la magnitud.

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 3.2</strong><br><br><b>1.</b> Al comparar los diagramas de dispersión original y estandarizado, ¿qué aspectos de la nube se mantienen y cuáles cambian en los ejes?<br><br><b>2.</b> ¿Qué muestran los histogramas o diagramas de caja sobre la forma y dispersión de cada característica? ¿Observas asimetrías o valores alejados?<br><br><b>3.</b> ¿Qué dirección e intensidad sugiere la correlación entre las entradas? ¿Qué patrones o conclusiones no pueden establecerse solo con estos gráficos?<br><br><em>Pista: describe evidencia visible, no causas; una correlación débil no descarta subgrupos y las etiquetas externas no se usan en esta exploración.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br><br> Tu respuesta aquí. Compara los dos diagramas y describe los patrones visibles en histogramas o cajas. Resume la correlación con su dirección e intensidad y señala una conclusión que estos gráficos no permiten afirmar.</div>

## 4. Comparar métodos y seleccionar la partición

### 4.1. Ajustar y evaluar cada método

En esta sección compararás dos métodos no supervisados sobre la misma matriz estandarizada:

- **K-Means** asigna cada observación al centroide más cercano y resume la dispersión intragrupo con WSS (Inercia). Su solución puede variar según la inicialización.
- **Clustering jerárquico aglomerativo** comienza con cada observación como un grupo y va fusionando grupos sucesivamente. Usaremos principalmente el enlace **Ward**, que en cada paso busca minimizar el aumento de la variación intragrupo. El dendrograma representa esas fusiones; al cortarlo para obtener un valor de $k$ se obtiene una partición.

**Sigue este orden; cada paso prepara el siguiente:**

1. Ajusta K-Means para todos los valores de `valores_k` y guarda en `resultados_kmeans[k]` el modelo y las etiquetas.
2. En la celda siguiente, recupera esas salidas y añade las métricas y la inercia. Comprueba que cada entrada contiene `modelo`, `grupos`, `metricas` e `inertia`.
3. Repite el proceso para Ward: primero guarda modelo y etiquetas en `resultados_ward`; después añade las métricas.
4. Cuando ambos diccionarios estén completos, pasa a 4.2 para reunirlos en una tabla. No intentes crear esa tabla antes de terminar los ajustes y las métricas.

K-Means proporciona `inertia_` (WSS); para Ward, usa el WSS calculado por `evaluar_particion`. Al calcular Silhouette, asegúrate de que la partición tenga entre 2 y $n-1$ grupos no vacíos. La interpretación conjunta de las métricas se hace después de reunir los resultados en la tabla.

**Funciones preparadas:** `ajustar_kmeans` y `ajustar_jerarquico` están en la última celda de código de la sección 1; ambas devuelven el modelo y las etiquetas de grupo. `evaluar_particion` está en la primera celda de código de esa sección y devuelve WSS y los índices internos.

**Dónde mirar:** [S5, matriz de distancias, enlaces y dendrogramas (5.2.1–5.2.3)](../sesion5/s5.ipynb); [S5, WSS, codo e índices internos (5.2.5–5.2.6)](../sesion5/s5.ipynb); [S5, cortes jerárquicos (5.2.7)](../sesion5/s5.ipynb) y [K-Means: ajuste y número de grupos (5.5.1–5.5.2)](../sesion5/s5.ipynb).

In [ ]:
# YA PREPARADO — NO MODIFICAR: fija los valores candidatos de k y prepara dónde guardar resultados.
valores_k = [2, 3, 4, 5, 6]  # Se compararán los mismos valores con K-Means y Ward.
resultados_kmeans = {}  # Guardará modelo, grupos, métricas e inercia de K-Means por k.
resultados_ward = {}  # Guardará modelo, grupos y métricas de Ward por k.

In [ ]:
# Recorre los valores candidatos; completa lo que falta dentro del bucle.
for k in valores_k:
    # TODO: ajusta K-Means sobre X_sintetico_std para este k y guarda las dos salidas en modelo y grupos.
    # Pista: ajustar_kmeans(X_sintetico_std, k) devuelve el modelo y las etiquetas de grupo.
    # TODO: guarda modelo y grupos en resultados_kmeans[k], con las claves "modelo" y "grupos".
    # Pista: crea un diccionario con pares clave: valor; en la siguiente celda añadirás las métricas.
    # E.G: resultados_kmeans[k] = ajustar_kmeans(...)
    pass  # Sustituye pass por tu código, manteniendo la indentación dentro del bucle.

In [ ]:
# Antes de continuar, completa la celda anterior para guardar modelo y grupos de cada k.
for k in valores_k:
    # TODO: recupera el modelo y los grupos de resultados_kmeans[k].
    # Pista: consulta resultados_kmeans[k]["modelo"] y resultados_kmeans[k]["grupos"].
    # TODO: calcula las métricas con evaluar_particion(X_sintetico_std, grupos).
    # TODO: añade al diccionario de este k las claves "metricas" e "inertia".
    # Pista: usa resultados_kmeans[k]["metricas"] = ...; para la inercia consulta modelo.inertia_.
    pass  # Sustituye pass por tu código, manteniendo la indentación dentro del bucle.

# Comprueba que cada k contiene "modelo", "grupos", "metricas" e "inertia" antes de continuar.

In [62]:
# Recorre los mismos valores de k; completa lo que falta dentro del bucle.
for k in valores_k:
    # TODO: ajusta clustering jerárquico con enlace Ward y guarda las salidas en modelo y grupos.
    # Pista: ajustar_jerarquico(X_sintetico_std, k, enlace="ward") devuelve modelo y etiquetas.
    # TODO: guarda modelo y grupos en resultados_ward[k], con las claves "modelo" y "grupos".
    # Pista: crea un diccionario con pares clave: valor, como hiciste con K-Means.
    pass  # Sustituye pass por tu código, manteniendo la indentación dentro del bucle.

In [63]:
# Antes de continuar, completa la celda anterior para guardar modelo y grupos de cada k.
for k in valores_k:
    # TODO: recupera los grupos de resultados_ward[k] usando la clave "grupos".
    # TODO: calcula evaluar_particion(X_sintetico_std, grupos) y guarda la serie
    # en resultados_ward[k] con la clave "metricas", sin borrar "modelo" ni "grupos".
    # Pista: resultados_ward[k]["metricas"] = ... añade la serie al diccionario existente.
    # Esa serie incluye el WSS jerárquico; el modelo no proporciona inertia_.
    pass  # Sustituye pass por tu código, manteniendo la indentación dentro del bucle.

# Comprueba que cada k contiene "modelo", "grupos" y "metricas" antes de continuar.

### 4.2. Reunir y comparar los resultados

Usa los resultados guardados en el apartado 4.1 para construir una tabla conjunta: cada fila debe identificar un método y un valor de $k$, e incluir WSS y las cuatro medidas internas. Esta parte organiza las salidas ya calculadas; no vuelvas a ajustar los modelos.

La tabla permite comparar K-Means y Ward para un mismo $k$, pero no sustituye al dendrograma ni a la inspección de tamaños. Compara tendencias: WSS y Davies–Bouldin se minimizan; Silhouette, Calinski–Harabasz y Dunn se maximizan. Las magnitudes de índices distintos no son directamente comparables.

**Referencias para interpretar las medidas** (documentación en inglés):

- **WSS o inercia:** [criterio de K-Means en scikit-learn](https://scikit-learn.org/stable/modules/clustering.html#k-means) y [atributo `inertia_`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html). En Ward, el WSS se calcula en esta práctica respecto a las medias de los grupos.
- **Silhouette:** [coeficiente Silhouette en scikit-learn](https://scikit-learn.org/stable/modules/clustering.html#silhouette-coefficient).
- **Calinski–Harabasz:** [índice Calinski–Harabasz en scikit-learn](https://scikit-learn.org/stable/modules/clustering.html#calinski-harabasz-index).
- **Davies–Bouldin:** [índice Davies–Bouldin en scikit-learn](https://scikit-learn.org/stable/modules/clustering.html#davies-bouldin-index).
- **Dunn:** en esta práctica lo calcula `dunn_index`, definido en la primera celda de código de la sección 1; scikit-learn no proporciona una función para este índice. Para entender las distancias utilizadas, consulta [pdist](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.pdist.html) y [cdist](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.cdist.html) de SciPy y [S5, índices internos (5.2.6)](../sesion5/s5.ipynb).

### Pasos y pistas para organizar la tabla conjunta

Cada diccionario de resultados (`resultados_kmeans` y `resultados_ward`) está organizado por $k$. En cada entrada, la clave `metricas` contiene la serie con WSS y los cuatro índices internos. Convierte esas series en filas y añade a cada fila las columnas `Metodo` y `k`.

Comprueba que aparece una fila por cada combinación de método y $k$, que se incluyeron todos los valores de `valores_k` y que la tabla contiene WSS más las cuatro medidas internas. Para K-Means, puedes recuperar WSS de `resultados_kmeans[k]["modelo"].inertia_`; para Ward, toma el WSS de `resultados_ward[k]["metricas"]["WSS"]`.

No ordenes todas las métricas como si compartieran dirección: WSS y Davies–Bouldin se minimizan, mientras Silhouette, Calinski–Harabasz y Dunn se maximizan. La tabla organiza la comparación; no determina por sí sola el $k$ ganador.

In [ ]:
# Prepara un diccionario para reunir las series de métricas de cada método.
series_por_metodo = {"K-Means": {}, "Ward": {}}

# Recorre los dos diccionarios de resultados y, dentro de cada uno, sus valores de k.
for metodo, resultados in [("K-Means", resultados_kmeans), ("Ward", resultados_ward)]:
    for k, resultado in resultados.items():
        # TODO: recupera la serie de métricas de resultado mediante la clave "metricas".
        # TODO: guarda esa serie en series_por_metodo[metodo][k].
        # Pista: cada serie contiene WSS y los cuatro índices internos; no vuelvas a ajustar los modelos.
        pass  # Sustituye pass por tu código, manteniendo la indentación.

In [ ]:
# TODO: usa series_por_metodo para crear una tabla por método, con una fila por valor de k.
# Pista: cada entrada series_por_metodo[metodo] es un diccionario de series, una por k.
# Para K-Means, empieza con tabla_kmeans = pd.concat(..., axis=1).T
# Completa ... con el diccionario del método: axis=1 reúne las series en columnas
# y .T las transpone para obtener una fila por k y una columna por métrica.
# Repite la operación para Ward y guarda el resultado en tabla_ward.


# TODO: añade a cada tabla la columna "Metodo" y pasa el índice de k a una columna llamada "k".
# Pista: añade .rename_axis("k").reset_index() a cada tabla para convertir el índice en columna.
# Después, usa tabla_kmeans["Metodo"] = ... y tabla_ward["Metodo"] = ...
# Completa cada asignación con el nombre del método correspondiente, entre comillas.
# Muestra ambas tablas con display(...) y consérvalas para concatenarlas en la siguiente celda.



In [ ]:
# TODO: concatena las dos tablas de métodos y ordena la tabla conjunta por "Metodo" y "k".
# Pista: pd.concat([tabla_kmeans, tabla_ward], ignore_index=True) une tablas por filas;
# después puedes usar .sort_values(["Metodo", "k"]). Adapta los nombres a tus tablas.
# Comprueba que hay una fila por método y por cada valor de valores_k; aquí organizas los resultados, no eliges todavía.



### Revisar errores y condiciones de las métricas

Antes de comparar Silhouette, verifica que las etiquetas contienen al menos dos grupos y que no hay un grupo por observación. En caso contrario, el índice puede no estar definido o no ser interpretable.

Para Dunn, el numerador es la menor distancia entre dos grupos y el denominador el mayor diámetro intragrupo; si un grupo contiene un solo punto, su diámetro se toma como cero. Revisa si aparecen valores no finitos y no los conviertas silenciosamente en cero.

El WSS depende de los centroides, por lo que su cálculo en una partición jerárquica es una suma descriptiva respecto a las medias, aunque Ward haya usado su propio criterio de fusión.

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 4.2</strong><br><br><b>1.</b> ¿Qué mide el WSS (Inercia) y por qué no basta con escoger el menor valor al comparar distintos números de grupos?<br><br><b>2.</b> ¿Qué aspectos de cohesión y separación recogen Silhouette, Calinski–Harabasz, Davies–Bouldin y Dunn? ¿En qué dirección se considera mejor cada índice?<br><br><b>3.</b> Si dos índices favorecen particiones distintas en tu tabla, ¿por qué puede ocurrir? ¿Qué limitación tiene usar solo métricas internas para evaluar la agrupación?<br><br><em>Pista: apóyate en la tabla para ilustrar tu explicación, pero no selecciones todavía el método ni el valor de k; esa decisión se justifica en el apartado 4.4.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br><br>Tu respuesta aquí. Explica qué miden el WSS y los cuatro índices, el sentido de cada uno y por qué pueden discrepar. Ilustra tu explicación con la tabla e indica una limitación de la evaluación interna.</div>

### 4.3. Leer el dendrograma y comparar enlaces

En un dendrograma, la altura representa la distancia a la que se fusionan dos observaciones o grupos. Para proponer un corte, identifica una altura candidata y cuenta las ramas que cruza una línea horizontal. Un salto vertical grande puede orientar la elección, pero no determina por sí solo el número correcto de grupos.

Como hay 1 000 observaciones, la función preparada dibuja un dendrograma resumido con las últimas 25 fusiones. Algunas hojas representan subárboles; el número bajo cada hoja indica cuántas observaciones contiene. Para los valores de $k$ considerados aquí, céntrate en las fusiones superiores y cuenta las ramas que quedan al cortar a esa altura.

Examina el dendrograma Ward preparado y dibuja el de enlace completo. Después compara Ward y enlace completo usando las mismas entradas y un mismo $k$. Ward busca limitar el aumento de variación intragrupo y usa distancia euclídea; el enlace completo define la distancia entre grupos por el par de observaciones más alejadas. Contrasta sus asignaciones, tamaños y métricas internas sin consultar las etiquetas externas. Compara las alturas dentro de cada dendrograma, no directamente entre enlaces.

**Funciones preparadas:** `dibujar_dendrograma_resumido` está junto a las funciones de ajuste de la sección 1; puedes elegir el enlace con `metodo="ward"` o `metodo="complete"`. `ajustar_jerarquico` y `evaluar_particion` también están preparadas en la sección 1.

In [ ]:
# YA PREPARADO — NO MODIFICAR: dibuja un resumen legible de las últimas fusiones Ward.
dibujar_dendrograma_resumido(X_sintetico_std, metodo="ward")  # Las hojas resumidas muestran cuántas observaciones contienen.

# TODO: dibuja el dendrograma resumido con enlace completo usando la misma matriz estandarizada.
# Pista: llama a dibujar_dendrograma_resumido(X_sintetico_std, metodo="complete").

In [ ]:
# TODO: observa el dendrograma Ward y elige una altura de corte candidata.
# Pista 1: guarda la matriz de fusiones en Z_ward = linkage(..., method="ward").
# Completa ... con la misma matriz estandarizada usada para dibujar el dendrograma.
# Cada fila de Z_ward representa una fusión y la columna de índice 2 contiene su altura.
# Pista 2: ultimas_alturas_ward = Z_ward[-5:, 2] extrae las cinco últimas alturas;
# muéstralas con print(...) y compáralas para localizar saltos grandes.
# Pista 3: imagina una línea horizontal entre dos alturas consecutivas, sin situarla
# exactamente sobre una fusión. Cuenta en el dendrograma cuántas ramas corta: ese es k.
# Si quieres explorar los intervalos para k entre 2 y 6, recorre valores_k con un bucle:
# for k in valores_k:
#     altura_inferior = Z_ward[-k, 2]
#     altura_superior = Z_ward[-(k - 1), 2]
#     print(...)  # Muestra k y ambas alturas; elige una altura entre ellas si son distintas.
# Explica en la respuesta 4.3 el intervalo o salto observado y el número de ramas.
# Es una propuesta provisional: contrástala después con las métricas y no uses etiquetas externas.



In [ ]:
# TODO: guarda un k_provisional de valores_k para comparar los enlaces, sin cerrar la selección final.
# TODO: recupera las etiquetas Ward para ese k de resultados_ward[k_provisional]["grupos"].
# TODO: ajusta enlace completo con el mismo k y guarda por separado su modelo y etiquetas.
# Pista: ajustar_jerarquico(X_sintetico_std, k_provisional, enlace="complete") devuelve modelo y etiquetas.
# Las funciones ajustar_jerarquico y dibujar_dendrograma_resumido están preparadas en la sección 1.



In [ ]:
# TODO: compara los tamaños de grupo de Ward y enlace completo para k_provisional.
# Pista de código: pd.Series(etiquetas).value_counts().sort_index() cuenta los puntos por grupo.
# Calcula los recuentos por separado; los identificadores de grupo son arbitrarios y pueden intercambiarse.


# TODO: cruza las dos asignaciones para observar cómo se reparten las mismas observaciones.
# Pista de código: pd.crosstab(etiquetas_ward, etiquetas_completo); adapta los nombres a tus variables.



In [ ]:
# TODO: compara las métricas internas de las dos particiones sobre X_sintetico_std.
# Pista: recupera resultados_ward[k_provisional]["metricas"] y usa evaluar_particion para las etiquetas de completo.
# Para mostrarlas lado a lado, usa pd.concat({"Ward": serie_ward, "Completo": serie_completo}, axis=1).
# La función evaluar_particion está definida en la primera celda de código de la sección 1.
# La selección final de método y k se justifica en 4.4; no uses etiquetas_externas para esta comparación.



<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 4.3</strong><br><br><b>1.</b> ¿Qué fusiones del dendrograma Ward destacan por su altura? ¿Qué corte candidato sugieren y por qué?<br><br><b>2.</b> Manteniendo el mismo k, ¿cómo cambia la partición al usar enlace completo en lugar de Ward? Cita diferencias en tamaños o métricas internas.<br><br><b>3.</b> ¿Por qué la altura de fusión no debe compararse directamente entre enlaces ni usarse como única evidencia para elegir k?<br><br><em>Pista: apóyate en el dendrograma y en resultados internos; no uses las etiquetas externas para seleccionar el corte.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br><br>Describe el salto o corte candidato observado en Ward. Compara con enlace completo para el mismo $k$, citando una diferencia en tamaños o métricas internas, y explica por qué la decisión no depende solo de la altura del dendrograma.</div>

### 4.4. Seleccionar la partición con evidencia interna

Usa la curva de WSS, la tabla de índices internos, el dendrograma y los tamaños de grupo para decidir el método y el valor de $k$. La curva de codo incluye $k=1$ como referencia; la tabla de índices compara $k\geq2$, porque Silhouette no está definido para un solo grupo. Busca el punto a partir del cual añadir grupos reduce la WSS menos intensamente. El codo es una heurística, no una regla exacta: contrástalo con los otros índices y explica los desacuerdos.

Aunque se generaron tres etiquetas de origen, no las uses para elegir. Evalúa si la evidencia interna apunta a una partición cercana a tres grupos, pero no supongas que el número de etiquetas generado sea automáticamente el $k$ correcto. Registra `nombre_metodo` y `k_elegido` antes de consultar las etiquetas externas; luego podrás recuperar la partición y perfilar sus grupos en 4.5.

In [ ]:
# TODO: calcula WSS de K-Means para la curva del codo, incluidos valores k=1 hasta el máximo probado.
# Pista: la tabla de índices usa valores_k (empieza en 2), pero Silhouette no está definido para k=1.
# Crea valores_k_codo = range(1, max(valores_k) + 1) y una lista inercias.
# Pista: una lista vacía se crea con inercias = []; guardará un WSS por cada k, en orden.
# La función ajustar_kmeans está en la última celda de código de la sección 1 y devuelve
# dos valores: el modelo ajustado y las etiquetas. Para esta curva necesitas el modelo.

# TODO: define valores_k_codo e inicializa inercias antes de ejecutar el bucle.

for k in valores_k_codo:
    # TODO: ajusta K-Means sobre X_sintetico_std para este k y guarda las dos salidas.
    # Pista: modelo, grupos = ajustar_kmeans(..., k); completa el primer argumento.
    # TODO: añade la inercia del modelo a la lista.
    # Pista: inercias.append(...); consulta el atributo inertia_ del modelo ajustado.
    pass  # Sustituye pass por tu código, manteniendo la indentación dentro del bucle.

# Comprueba con len(...) que inercias y valores_k_codo tienen la misma longitud.
# No añadas k=1 a la tabla de índices internos; utiliza la lista en el gráfico siguiente.

In [ ]:
# TODO: representa WSS frente a valores_k_codo con puntos, título y etiquetas de ejes.
# Pista: plt.plot(valores_k_codo, inercias, marker="o") dibuja la curva si ambas listas están en el mismo orden.
# Usa plt.xlabel(...), plt.ylabel(...), plt.title(...) y plt.show() para completar el gráfico.
# Describe dónde la reducción de WSS empieza a ser menos acusada; el codo orienta, pero no decide por sí solo.

In [ ]:
# TODO: compara los candidatos usando la curva WSS, la tabla de índices y el dendrograma.
# Pista: filtra la tabla conjunta del apartado 4.2 con tabla[tabla["k"] == k] para comparar ambos métodos;
# sustituye tabla por el nombre que hayas usado y muestra las columnas de índices con .loc[:, [...]].
# Usa .sort_values("Silhouette", ascending=False) o .sort_values("Davies-Bouldin") para explorar cada índice.
# Contrasta los resultados con el posible codo y el dendrograma; no decidas por un único índice.
# Si representas las asignaciones, utiliza los grupos guardados; no colorees con etiquetas_externas.



In [ ]:
# TODO: revisa los tamaños de grupo para los métodos y valores de k guardados.
for metodo, resultados in [("K-Means", resultados_kmeans), ("Ward", resultados_ward)]:
    for k in valores_k:
        # TODO: recupera las etiquetas de grupo desde resultados[k]["grupos"].
        # TODO: cuenta los puntos por grupo y muestra el método, k y los recuentos.
        # Pista: pd.Series(grupos).value_counts().sort_index() cuenta los puntos por grupo.
        pass  # Sustituye pass por tu código, manteniendo la indentación.
# Los identificadores son arbitrarios: el grupo 0 de un método no tiene por qué coincidir con el del otro.

In [ ]:
# TODO: elige el método y el k final con varias evidencias internas y registra nombre_metodo y k_elegido.
# Pista: asigna a nombre_metodo una cadena que identifique "K-Means" o "Ward" y a k_elegido un valor de valores_k.
# Muestra ambos con print(...) y justifica la decisión en la respuesta 4.4 usando la tabla, la curva y los tamaños.
# La recuperación de las etiquetas finales y el cálculo de ARI se hacen en la sección 4.5.
# No consultes etiquetas_externas para cerrar esta selección.



<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 4.4</strong><br><br><b>1.</b> ¿Qué posible codo observas en la curva WSS? ¿Qué sugieren el dendrograma y los índices internos para los distintos valores de $k$?<br><br><b>2.</b> ¿Qué diferencias hay entre K-Means y Ward para el mismo $k$? Considera también los tamaños de grupo.<br><br><b>3.</b> ¿Qué método y valor de $k$ eliges finalmente? Justifica la decisión con varias evidencias internas e indica una limitación. No uses las etiquetas externas para seleccionar.<br><br><em>Pista: WSS y Davies–Bouldin se minimizan; Silhouette, Calinski–Harabasz y Dunn se maximizan. Busca tendencias y desacuerdos; no compares directamente las escalas de índices distintos.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br><br>Tu respuesta aquí. Indica `nombre_metodo` y `k_elegido`. Justifica la decisión citando el codo de WSS, índices internos, dendrograma o tamaños de grupo; explica cualquier desacuerdo y una limitación. Confirma que no usaste las etiquetas externas para elegir.</div>

### 4.5. Perfilar la partición elegida y hacer la comparación externa

Con `nombre_metodo` y `k_elegido` ya fijados en el apartado 4.4, recupera la partición seleccionada y describe sus tamaños y perfiles usando las características originales. No cambies la selección a partir de estos perfiles ni del ARI.

Después de cerrar y documentar la decisión interna, calcula el ARI frente a `etiquetas_externas`. Este índice compara la partición elegida con las etiquetas conocidas del conjunto sintético; sirve como comprobación externa, no para volver atrás y escoger el método o $k$.

**Orden de trabajo:**

1. Recupera las etiquetas de la configuración elegida en 4.4.
2. Resume el tamaño de cada grupo y sus medias o medianas por característica en unidades originales.
3. Representa la partición elegida y describe diferencias entre perfiles sin atribuirles causas.
4. Por último, calcula e interpreta el ARI como acuerdo entre particiones.

**Dónde mirar:** [S5, evaluación interna y externa](../sesion5/s5.ipynb).

In [ ]:
# TODO: recupera en clusters_grupos las etiquetas de la configuración fijada en el apartado 4.4.
# Pista: según nombre_metodo, selecciona resultados_kmeans o resultados_ward con un if / elif.
# Dentro del diccionario elegido, accede al valor de k_elegido y después a la clave "grupos".
# Por ejemplo, resultados_kmeans[k_elegido]["grupos"] recupera las etiquetas de K-Means.
# Comprueba con len(clusters_grupos) que hay una asignación por fila de X_sintetico.
# No uses etiquetas_externas para recuperar ni cambiar la partición.

In [ ]:
# TODO: muestra cuántas observaciones hay en cada grupo de clusters_grupos.
# Pista: pd.Series(clusters_grupos).value_counts().sort_index() cuenta y ordena las etiquetas.
# Comprueba que la suma de los tamaños coincide con el número de filas de X_sintetico.


In [ ]:
# TODO: resume las características originales de cada grupo con medias o medianas.
# Pista: X_sintetico.groupby(clusters_grupos) agrupa las filas usando las etiquetas seleccionadas.
# Consulta .mean() o .median() sobre ese agrupamiento y muestra la tabla con display(...).
# Compara las coordenadas de cada perfil en sus unidades originales; los identificadores son arbitrarios.



In [ ]:
# TODO: dibuja la partición seleccionada con las dos características originales.
# Pista: plt.scatter(X_sintetico.iloc[:, 0], X_sintetico.iloc[:, 1], c=clusters_grupos, cmap="tab10", alpha=0.6)
# colorea según los grupos obtenidos, no según etiquetas_externas. Añade título y etiquetas de ejes con
# plt.title(...), plt.xlabel(...), plt.ylabel(...) y muestra el gráfico con plt.show().
# Relaciona el gráfico con los tamaños y perfiles calculados; describe patrones sin atribuirles causas.



### Lectura: índice de Rand ajustado (ARI)

**Antes de calcular el ARI**, lee el apartado [Rand index de la documentación oficial de scikit-learn](https://scikit-learn.org/stable/modules/clustering.html#adjusted-rand-score) (en inglés). Presta atención a cómo compara dos particiones, al ajuste por azar y a por qué cambiar los números de las etiquetas no altera el resultado.

Como orientación, un ARI de **1** indica particiones idénticas salvo por los nombres de los grupos; un valor cercano a **0** corresponde aproximadamente al acuerdo esperado por azar y puede ser negativo. No es un porcentaje de aciertos. En esta práctica, consulta las etiquetas de origen solo después de cerrar la selección interna y no cambies la decisión a partir del ARI.

**Función preparada:** `ari_externo` está definida en la **primera celda de código de la sección 1**. Recibe las etiquetas de referencia y las etiquetas de la partición seleccionada, en ese orden, y devuelve el ARI mediante `adjusted_rand_score` de scikit-learn. En la celda siguiente úsala con `etiquetas_externas` y `clusters_grupos`, después de cerrar la selección.

In [ ]:
# TODO: después de cerrar la selección interna, calcula el ARI frente a etiquetas_externas.
# Pista: ari_externo(etiquetas_externas, clusters_grupos) devuelve el acuerdo entre las dos particiones.
# Guarda y muestra el resultado con print(...); ari_externo está en la primera celda de código de la sección 1.
# El ARI no es exactitud predictiva: no cambies nombre_metodo ni k_elegido a partir de este valor.

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 4.5.1</strong><br><br><b>1.</b> ¿Cuántas observaciones hay en cada grupo? ¿Qué diferencias muestran sus medias o medianas para cada característica?<br><br><b>2.</b> ¿Cómo se reflejan esos perfiles en el gráfico? ¿Qué describen y qué no permiten afirmar por sí solos?<br><br><em>Pista: cita valores de las características originales y describe patrones, no causas; los identificadores de grupo son arbitrarios.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br><br>Tu respuesta aquí. Cita tamaños y al menos dos diferencias entre los perfiles. Explica qué describen y menciona una conclusión que no se puede extraer de esta partición exploratoria.</div>

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 4.5.2</strong><br><br><b>1.</b>¿Qué ARI obtuviste para la partición seleccionada y qué indica sobre su acuerdo con las etiquetas de origen?<br><br><b>2.</b> ¿Por qué se calcula después de cerrar la selección? ¿Qué no permite concluir por sí solo?<br><br><em>Pista: el ARI es una comparación externa entre particiones; no es exactitud predictiva ni un criterio para cambiar la elección.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br><br>Tu respuesta aquí. Informa e interpreta el ARI como acuerdo entre la partición seleccionada y las etiquetas externas. Explica por qué se calcula después de cerrar la selección y qué no permite afirmar.</div>

## 5. Preparar clasificación de dígitos

Cada observación representa una imagen de un dígito manuscrito de 8 × 8 píxeles. Las 64 variables son intensidades numéricas; la respuesta toma valores de 0 a 9. Compararemos métodos vistos en S6 y S7: **KNN, Naive Bayes, árbol de decisión, random forest y SVM**. No se incluirá regresión logística, ya que la vimos en la práctica anterior.

Primero estratificamos los datos en ajuste, validación y test, manteniendo aproximadamente el balance de cada cifra. El test no participará en el preprocesado, la búsqueda ni la selección.

**Dónde mirar:** [S6, KNN: partición, escalas y evaluación (6.1)](../sesion6/s6.ipynb) y [S6, Naive Bayes: ajuste, posteriores y métricas por clase (6.2)](../sesion6/s6.ipynb). Para árboles, bosques y SVM, consulta [S7, apartados 7.1–7.3](../sesion7/s7.ipynb).

### Por qué hay varios clasificadores

Los cinco candidatos representan reglas diferentes para clasificar píxeles:

- **KNN:** vota entre observaciones de entrenamiento cercanas; la escala puede cambiar qué significa «cerca».
- **Naive Bayes gaussiano:** combina distribuciones por clase y supone independencia condicional de las entradas.
- **Árbol:** construye reglas de corte interpretables, pero puede ser sensible a cambios de muestra.
- **Random forest:** promedia árboles y suele reducir la variabilidad de un árbol individual.
- **SVM RBF:** busca una frontera flexible; el escalado se mantiene en su pipeline.

La comparación usa los mismos ejemplos y métricas. La lista no presupone que un método vaya a ganar.

### 5.1. Preparar y explorar las particiones

Ejecuta la celda preparada para crear particiones estratificadas y reproducibles. Completa el `TODO` para comparar el reparto de las cifras y comenta una limitación del conjunto en la pregunta de datos.

KNN y SVM aprenderán la escala dentro del pipeline usando solo los datos de entrenamiento de cada pliegue. Mantén test apartado: no lo uses para calcular escalas, ajustar parámetros ni elegir modelos.

In [ ]:
# YA PREPARADO — NO MODIFICAR: crea divisiones estratificadas reproducibles.
X_dig_train_valid, X_dig_test, y_dig_train_valid, y_dig_test = train_test_split(
    X_digitos, y_digitos, test_size=0.20, random_state=SEMILLA, stratify=y_digitos
)  # Reserva test con proporciones similares de cada cifra.
X_dig_train, X_dig_valid, y_dig_train, y_dig_valid = train_test_split(
    X_dig_train_valid, y_dig_train_valid, test_size=0.25,
    random_state=SEMILLA, stratify=y_dig_train_valid
)  # Deja aproximadamente 60 % ajuste, 20 % validación y 20 % test.
print("Tamaños:", X_dig_train.shape, X_dig_valid.shape, X_dig_test.shape)

In [ ]:
# TODO: compara los recuentos y las proporciones por dígito en ajuste, validación y test.
# Pista: y_dig_train.value_counts().sort_index() cuenta las cifras en ajuste;
# repite la operación con y_dig_valid e y_dig_test para comprobar que las clases están representadas.
# Para proporciones, utiliza .value_counts(normalize=True).sort_index() en cada partición.
# Puedes reunir las tres series en un DataFrame y comparar sus porcentajes; los tamaños son distintos.
# Usa los recuentos de test solo para revisar el reparto, nunca para elegir modelos o parámetros.

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 5.1</strong><br><br><b>1.</b> ¿Están las diez cifras representadas en ajuste y validación? ¿Qué información codifica cada fila?<br><br><b>2.</b> ¿Qué limitación del conjunto puede afectar a la generalización de los resultados?<br><br><em>Pistas: los datos contienen 1 797 imágenes de solo 8 × 8 píxeles. Piensa en cómo pudieron seleccionarse las personas y ejemplos: ¿representan distintos estilos de escritura, edades, lugares o dispositivos? ¿Qué tipos de escritura podrían faltar? La estratificación conserva las proporciones de cifras en estas particiones, pero no garantiza que la muestra original represente a toda la población. Basa tu respuesta en lo que sabes del conjunto y evita afirmar características de la muestra que no puedas comprobar.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br><br>Tu respuesta aquí. Cita los recuentos, describe qué representa cada fila y explica una limitación del conjunto.</div>

### 5.2. Comprobar la separación de las particiones

En 5.1 revisaste la distribución de las cifras; ahora comprueba que las particiones no comparten índices de fila. La celda preparada verifica cada pareja. Después informa cuántos índices coinciden y explica una limitación: índices distintos no garantizan que dos imágenes no tengan los mismos valores de píxel.

Esta comprobación usa los índices para verificar el reparto, no los resultados de test para ajustar o seleccionar modelos.

In [ ]:
# YA PREPARADO — NO MODIFICAR: verifica que los índices de las particiones no se solapan.
assert X_dig_train.index.intersection(X_dig_valid.index).empty
assert X_dig_train.index.intersection(X_dig_test.index).empty
assert X_dig_valid.index.intersection(X_dig_test.index).empty

In [ ]:
# TODO: informa cuántos índices comparten cada pareja de particiones y comenta qué comprueba el resultado.
# Pista: X_dig_train.index.intersection(X_dig_valid.index) devuelve los índices comunes de ajuste y validación.
# Usa len(...) para contarlos y repite la operación para ajuste/test y validación/test.
# Compara los recuentos con las comprobaciones preparadas de arriba y explica qué significa que sean cero.
# Un índice diferente identifica otra fila, pero no descarta que dos filas tengan los mismos valores de píxel.

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 5.2</strong><br><br><b>1.</b> ¿Cuántos índices comparten ajuste y validación, ajuste y test, y validación y test? ¿Qué significa que las tres intersecciones estén vacías?<br><br><b>2.</b> ¿Qué comprueba este resultado y qué no permite descartar sobre el contenido de las imágenes?<br><br><em>Pista: cita los recuentos del TODO. Un índice identifica una fila del conjunto original; filas distintas pueden contener los mismos píxeles.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br><br>Tu respuesta aquí. Indica los tres recuentos de índices comunes. Explica qué confirma que sean cero y aclara que índices distintos no descartan filas con imágenes de píxeles idénticos.</div>

### 5.3. Crear una referencia sencilla

Antes de comparar modelos, ejecuta la celda preparada para establecer una **línea base**: predecir siempre la cifra más frecuente del conjunto de ajuste. Aunque no utiliza los píxeles, sirve como referencia para valorar los clasificadores.

La clase mayoritaria se determina solo con ajuste; la regla se evalúa en validación. Mantén test apartado para la evaluación final.

**Pistas para interpretar la referencia:** observa la cifra mayoritaria y compara la exactitud con el F1 macro. En la matriz, las filas representan predicciones y las columnas, clases reales; revisa qué filas reciben predicciones y qué ocurre con las demás clases. Consulta la [documentación de scikit-learn sobre F1](https://scikit-learn.org/stable/modules/model_evaluation.html#f1-score) para su definición y el promedio multiclase.


In [ ]:
# YA PREPARADO — NO MODIFICAR: evalúa una regla que predice siempre la clase más frecuente en ajuste.
clase_mayoritaria = int(y_dig_train.mode().iloc[0])  # Usa solo las etiquetas de ajuste.
pred_baseline_valid = np.full(len(y_dig_valid), clase_mayoritaria)  # Predice la misma clase para cada fila de validación.
metricas_baseline = metricas_multiclase(y_dig_valid, pred_baseline_valid)
print("Clase mayoritaria en ajuste:", clase_mayoritaria)
display(metricas_baseline)  # Muestra las métricas de validación.
clases_digitos = sorted(y_digitos.unique())
matriz_baseline = matriz_confusion_multiclase(y_dig_valid, pred_baseline_valid, clases_digitos)
display(pd.DataFrame(matriz_baseline, index=clases_digitos, columns=clases_digitos).rename_axis(index="Predicción", columns="Real"))

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 5.3</strong><br><br><b>1.</b> ¿Qué cifra predice la regla mayoritaria? ¿Qué exactitud y F1 macro obtiene en validación?<br><br><b>2.</b> ¿Por qué conviene comparar los clasificadores con esta referencia? ¿Qué muestra la matriz sobre las clases que no predice?<br><br><em>Pista: cita las métricas y lee la matriz con predicciones en filas y clases reales en columnas. Compara con los candidatos cuando los hayas evaluado.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br><br>Tu respuesta aquí. Indica la clase predicha, cita las métricas de validación e interpreta la matriz. Compara con los candidatos después de evaluarlos.</div>

### 5.4. Comparar dos imágenes con una distancia

La galería muestra cómo se representan las cifras mediante 64 intensidades. Elige dos imágenes y calcula la distancia euclídea entre sus filas: es una forma sencilla de cuantificar cuánto difieren sus píxeles y sirve para introducir la idea de proximidad de KNN.

Usa dos posiciones de la galería (de 0 a 9), ambas del conjunto de ajuste.

In [ ]:
# YA PREPARADO — NO MODIFICAR: muestra diez imágenes del conjunto de ajuste.
fig, ejes = plt.subplots(2, 5, figsize=(9, 4))  # Reserva diez paneles para ver cifras distintas.
for ax, indice in zip(ejes.flat, range(10)):  # Recorre paneles y selecciona ejemplos de ajuste.
    fila = X_dig_train.iloc[indice].to_numpy().reshape(8, 8)  # Reconstruye la matriz de píxeles.
    ax.imshow(fila, cmap="gray_r", interpolation="nearest")  # Dibuja intensidades claras y oscuras.
    ax.set_title(f"Etiqueta: {y_dig_train.iloc[indice]}")  # Añade la cifra verdadera de esta imagen.
    ax.axis("off")  # Oculta coordenadas que no aportan al ejercicio.
fig.tight_layout()  # Reduce espacios entre los diez dibujos.
plt.show()  # Presenta la galería de ejemplos.

In [ ]:
# TODO: compara dos imágenes de la galería mediante la distancia euclídea.
indice_a = 0  # Cambia por una posición de la galería entre 0 y 9.
indice_b = 1  # Elige otra posición distinta entre 0 y 9.
# Pista: toma las filas con X_dig_train.iloc[indice_a] e iloc[indice_b].
# Usa np.linalg.norm(...) sobre la diferencia entre las dos filas para calcular la distancia.
# Muestra también ambas etiquetas con y_dig_train.iloc[...] para interpretar el resultado.

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 5.4</strong><br><br><b>1.</b> ¿Qué distancia obtuviste para las dos imágenes y qué etiquetas tienen?<br><br><b>2.</b> ¿La distancia concuerda con el parecido visual? ¿Qué nos enseña este ejemplo sobre la proximidad que usa KNN?<br><br><em>Pista: una distancia menor indica píxeles más parecidos según esta medida, pero no garantiza por sí sola que las etiquetas sean iguales.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br><br>Tu respuesta aquí. Indica las posiciones y etiquetas elegidas, cita la distancia y relaciónala con el parecido visual y la idea de proximidad en KNN.</div>

## 6. Comparar y seleccionar clasificadores

Compara KNN, Naive Bayes gaussiano, árbol, random forest y SVM con las mismas particiones. La búsqueda de parámetros se hace mediante validación cruzada estratificada dentro de ajuste; la validación externa sirve para comparar y elegir el modelo. El test queda reservado para la sección 7.

El escalado de KNN y SVM se mantiene dentro de sus pipelines para que cada pliegue aprenda la transformación solo con sus datos de ajuste. No estandarices todo el conjunto antes de dividirlo.

**Dónde mirar:** [S6, KNN y escalado (6.1)](../sesion6/s6.ipynb) y [Naive Bayes y evaluación por clase (6.2)](../sesion6/s6.ipynb); [S7, árboles y poda (7.1)](../sesion7/s7.ipynb), [random forest (7.2)](../sesion7/s7.ipynb) y [SVM con escalado y búsqueda de hiperparámetros (7.3)](../sesion7/s7.ipynb).

### 6.1. Ajustar candidatos y comparar resultados

Todos los candidatos reciben las mismas filas. Se seleccionan los hiperparámetros con F1 macro en la validación cruzada de ajuste; después se comparan las métricas en validación externa. Consulta la [documentación de scikit-learn sobre F1](https://scikit-learn.org/stable/modules/model_evaluation.html#f1-score) para su definición y el promedio multiclase.

**Funciones auxiliares preparadas en la sección 1 (primera celda de código):** `crear_busqueda_clasificador` construye la búsqueda con validación cruzada; `metricas_multiclase` calcula las métricas de validación; y `reunir_series_metricas` organiza las series en una tabla comparativa. Úsalas en las celdas siguientes para ajustar y evaluar los candidatos. La elección final se hará en 6.2, después de revisar las métricas; no uses test en esta fase.

In [ ]:
# YA PREPARADO — NO MODIFICAR: define candidatos.
candidatos = {
    "KNN": (
        Pipeline([("escala", StandardScaler()), ("modelo", KNeighborsClassifier())]),
        {"modelo__n_neighbors": [3, 5, 9]},
    ),
    "Naive Bayes gaussiano": (
        GaussianNB(), {"var_smoothing": [1e-9, 1e-8, 1e-7]}
    ),
    "Árbol": (
        DecisionTreeClassifier(random_state=SEMILLA),
        {"max_depth": [5, 10, None], "min_samples_leaf": [1, 3]},
    ),
    "Random forest": (
        RandomForestClassifier(n_estimators=100, random_state=SEMILLA, n_jobs=1),
        {"max_features": ["sqrt", 0.5], "min_samples_leaf": [1, 2]},
    ),
    "SVM RBF": (
        Pipeline([("escala", StandardScaler()), ("modelo", SVC())]),
        {"modelo__C": [1, 10], "modelo__gamma": ["scale"]},
    ),
}

# YA PREPARADO: diccionarios para conservar los resultados de cada candidato.
busquedas = {}
mejores_estimadores = {}
mejores_parametros = {}
puntuaciones_cv = {}
metricas_validacion = {}

# TODO: recorre candidatos con .items(); cada vuelta entrega nombre, estimador y rejilla.
for nombre, (estimador, rejilla) in candidatos.items():
    # TODO: crea la búsqueda con crear_busqueda_clasificador(estimador, rejilla).
    # Guarda el objeto en una variable llamada busqueda, i.e. busqueda = crear_busqueda_clasificador(...)

    # TODO: ajusta la búsqueda solo con X_dig_train e y_dig_train con busqueda.fit(...)
    # No uses X_dig_valid ni X_dig_test para ajustar o buscar parámetros.

    # TODO: guarda la búsqueda en busquedas[nombre] y conserva estos atributos:
    # best_estimator_ en mejores_estimadores[nombre],
    # best_params_ en mejores_parametros[nombre],
    # best_score_ en puntuaciones_cv[nombre].
    pass  # Sustituye pass por las operaciones indicadas.

In [ ]:
# TODO: evalúa en validación el mejor estimador encontrado para cada candidato.
for nombre, mejor_estimador in mejores_estimadores.items():
    # Predice las etiquetas de X_dig_valid con mejor_estimador.predict(...).
    # Calcula metricas_multiclase usando y_dig_valid y esas predicciones.
    # Guarda la serie resultante en metricas_validacion[nombre].
    pass  # Sustituye pass por las operaciones indicadas.

In [ ]:
# TODO: combina las series de metricas_validacion en una tabla, un candidato por fila.
# Pista: Utiliza la funcion reunir_series_metricas definida al principio del notebook.
# Guarda la tabla resultante en tabla_validacion, i.e. tabla_validacion = reunir_series_metricas(...)



In [ ]:
# TODO: añade a tabla_validacion una columna llamada "F1 macro CV".
# Pista: pd.Series(puntuaciones_cv) alinea cada puntuación con el nombre de su candidato.
# Con tabla_validacion["nombre_columna"] = ... guardamos esa columna en la tabla.



In [ ]:
# TODO: muestra la tabla ordenada de mayor a menor por "F1 macro".
# Los resultados sirven para comparar candidatos; la elección razonada va en 6.2.



<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 6.1</strong><br><br><b>1.</b> ¿Qué candidatos obtienen el mayor y el menor F1 macro en validación externa? ¿Qué diferencia hay entre sus resultados?<br><br><b>2.</b> Para al menos dos candidatos, compara su F1 macro de validación cruzada con el de validación externa. ¿Qué papel tiene cada evaluación en esta práctica?<br><br><em>Pista: cita valores de la tabla y distingue «F1 macro CV» de «F1 macro». Describe la comparación sin elegir todavía el clasificador; esa decisión se justifica en 6.2. No consultes test.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br><br>Tu respuesta aquí. Identifica los extremos de F1 macro en validación externa y su diferencia. Compara los valores CV y de validación externa de al menos dos candidatos y explica para qué se usa cada evaluación.</div>

### 6.2. Elegir el clasificador e interpretar sus errores

Revisa la tabla de 6.1 y escoge un candidato considerando F1 macro, exactitud, sensibilidad macro y sencillez. Guarda su nombre exacto en `modelo_clasificador_elegido`; después recupera su estimador de `mejores_estimadores` y úsalo para predecir `X_dig_valid` y construir la matriz de confusión.

La **exactitud** resume la proporción total de aciertos. La **sensibilidad macro**, la **precisión macro** y el **F1 macro** calculan primero la métrica por cifra y luego promedian dando el mismo peso a cada clase. La matriz complementa esas métricas: su diagonal recoge aciertos y las celdas fuera de ella muestran confusiones. Identifica uno o dos pares y relaciónalos con exactitud y F1 macro. Las filas son predicciones y las columnas, cifras reales; no uses test en este análisis.

**Funciones y objetos preparados:** `matriz_confusion_multiclase` está definida en la segunda celda de código de la sección 1 y recibe, en este orden, las etiquetas reales, las predichas y la lista de clases. El método `predict` pertenece al estimador elegido, guardado en `mejores_estimadores` durante el apartado 6.1. Puedes ordenar las clases con `sorted` y mostrar la matriz en un `pd.DataFrame`; pandas se importó en la primera celda de código de la sección 1.

In [ ]:
# TODO: compara la tabla de 6.1 y elige una clave de candidatos.
# Pista: consulta los nombres disponibles con candidatos.keys() y compara sus métricas.
# Asigna el nombre elegido como cadena: modelo_clasificador_elegido = "..."
# Sustituye ... por una clave exacta de candidatos, respetando mayúsculas y espacios.
# Puedes comprobar la elección con: modelo_clasificador_elegido in ...
# Justifica la decisión en la respuesta 6.2 considerando métricas y sencillez; no uses test.



In [ ]:
# TODO: recupera el estimador elegido desde mejores_estimadores y guárdalo en una variable.
# Pista: estimador_elegido = mejores_estimadores[...]
# Sustituye ... por la variable que contiene la clave seleccionada en la celda anterior.
# Si aparece KeyError, comprueba el nombre elegido y que ejecutaste los ajustes de 6.1.



In [ ]:
# TODO: predice las etiquetas de validación y guárdalas en pred_valid_elegido.
# Pista: pred_valid_elegido = estimador_elegido.predict(...)
# Completa el argumento con las características de validación, no con sus etiquetas.
# El estimador ya está ajustado: no llames a fit con validación ni uses test.
# Comprueba los tamaños comparando len(pred_valid_elegido) con len(...).



In [ ]:
# TODO: crea la lista ordenada de clases (0–9) y guárdala en clases_valid.
# Pista: clases_valid = sorted(SERIE_ETIQUETAS.unique())
# Sustituye SERIE_ETIQUETAS por la variable que contiene las etiquetas de dígitos.



In [ ]:
# TODO: calcula la matriz de confusión de validación y guárdala en matriz_valid.
# Pista: matriz_valid = matriz_confusion_multiclase(..., ..., clases_valid)
# Completa los dos primeros argumentos con etiquetas reales y predichas, en ese orden.
# La función está definida en la segunda celda de código de la sección 1.



In [ ]:
# TODO: muestra la matriz en una tabla con las cifras como índice y columnas.
# Pista: tabla_matriz = pd.DataFrame(..., index=clases_valid, columns=...)
# Completa con la matriz y la lista de clases; después usa display(tabla_matriz).
# Puedes nombrar los ejes con .rename_axis(index="Predicción", columns="Real").
# Comprueba que la suma de la tabla coincide con el número de ejemplos de validación;
# revisa la diagonal y las celdas fuera de ella antes de responder 6.2.



<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 6.2</strong><br><br><b>1.</b> ¿Qué clasificador eliges y por qué? Justifica tu decisión con resultados de validación y considera su sencillez.<br><br><b>2.</b> En la matriz de validación del modelo elegido, ¿qué confusión o confusiones destacan? Indica las cifras reales, las predichas y sus recuentos. ¿Qué información añade la matriz a las métricas globales?<br><br><em>Pista: la fila es la predicción y la columna la cifra real. Si hay empate, menciona alguna de las confusiones empatadas. No uses test para elegir.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br><br>Indica el clasificador elegido y justifica la decisión con resultados de validación y sencillez. Cita una o más confusiones de su matriz, distinguiendo cifras reales y predichas y sus recuentos, y explica qué añade la matriz a las métricas globales.</div>

### 6.3. Conceptos de los clasificadores de S6 y S7

Responde brevemente a partir de las sesiones. No hace falta volver a ajustar modelos: algunos conceptos de esta pregunta, posteriores de Naive Bayes, evaluación OOB, poda y kernels, se repasan en el material enlazado.

**Consulta:** [S6, KNN y Naive Bayes](../sesion6/s6.ipynb); [S7, árboles y poda (7.1)](../sesion7/s7.ipynb); [S7, random forest y OOB (7.2)](../sesion7/s7.ipynb); [S7, SVM (7.3)](../sesion7/s7.ipynb).

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 6.3</strong><br><br><b>1. KNN y Naive Bayes.</b> ¿Cómo pueden influir la escala de los píxeles y el número de vecinos en KNN? ¿Qué supuesto hace Naive Bayes gaussiano y qué representa una probabilidad posterior?<br><br><b>2. Árboles y random forest.</b> ¿Cómo ayudan limitar la profundidad o podar un árbol? ¿Qué estima el error OOB y en qué se diferencia de la validación externa?<br><br><b>3. SVM.</b> ¿Qué diferencia cabe esperar entre un kernel lineal y uno RBF? ¿Por qué se mantiene el escalado dentro del pipeline?<br><br><em>Pista: relaciona cada idea con el funcionamiento del método; distingue la evaluación interna de un bosque de la evaluación sobre un conjunto de validación.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br><br>Tu respuesta aquí. Contesta brevemente los tres apartados y usa los conceptos de las sesiones, no solo el resultado de la tabla.</div>

## 7. Reajuste final y evaluación en test

Una vez fijados el método y sus hiperparámetros mediante validación, combina ajuste y validación y vuelve a ajustar. Evalúa el test una sola vez. Reporta exactitud, F1 macro, precisión y sensibilidad macro, además de la matriz de confusión.

En la matriz, las **filas son las cifras predichas** y las **columnas las cifras reales**. Observa qué dígitos se confunden; no modifiques el modelo a partir de ese análisis.

**Para cerrar el análisis:**

1. Recupera el estimador ganador y reajústalo con train + validación.
2. Predice test una sola vez; calcula las métricas y la matriz con clases en orden 0–9.
3. Inspecciona algunos errores y, si ayuda, vuelve a mostrar sus imágenes.
4. Resume las confusiones importantes y limita la conclusión a estos datos y esta partición.

**Funciones y objetos que necesitarás:**

- `copiar_mejor_estimador`: definida en la **primera celda de código de la sección 1**. Recibe la búsqueda del modelo elegido y devuelve una copia sin ajustar de su mejor estimador, con los hiperparámetros seleccionados. Las búsquedas se guardaron en `busquedas` en el apartado 6.1 y el nombre elegido en `modelo_clasificador_elegido` en 6.2.
- `fit` y `predict`: son métodos del estimador. Usa `fit` para ajustar la copia con train + validación y `predict` para obtener las etiquetas de test. Las tablas combinadas ya se preparan en la primera celda de código de esta sección.
- `metricas_multiclase`: definida en la **primera celda de código de la sección 1**. Recibe etiquetas reales y predichas, en ese orden, y devuelve exactitud, sensibilidad macro, precisión macro y F1 macro.
- `matriz_confusion_multiclase`: definida en la **segunda celda de código de la sección 1**. Recibe etiquetas reales, predichas y la lista ordenada de clases; devuelve predicciones en filas y valores reales en columnas. Ya la usaste en 6.2.
- `pd.DataFrame` y `plt.imshow`: sirven para mostrar la matriz como tabla y como gráfico; pandas y Matplotlib se importaron como `pd` y `plt` en la **primera celda de código de la sección 1**.

**Referencia:** [S4, matrices y métricas de clasificación](../sesion4/s4.ipynb) sirve para repasar exactitud, sensibilidad y precisión.

In [ ]:
# YA PREPARADO — NO MODIFICAR: combina las filas de ajuste y validación para el ajuste final.
X_dig_train_final = pd.concat([X_dig_train, X_dig_valid], axis=0)
y_dig_train_final = pd.concat([y_dig_train, y_dig_valid], axis=0)

In [ ]:
# TODO: recupera la búsqueda del modelo elegido y copia su mejor estimador en estimador_final.
# Pista: busquedas[...] permite acceder a la búsqueda usando la clave guardada en 6.2.
# Después, usa estimador_final = copiar_mejor_estimador(...), completando el argumento
# con esa búsqueda. La función está en la primera celda de código de la sección 1.
# La copia conserva los hiperparámetros elegidos, pero todavía no está ajustada.



In [ ]:
# TODO: ajusta estimador_final con los datos combinados de train y validación.
# Pista: estimador_final.fit(..., ...)
# Completa los argumentos con las características y las etiquetas combinadas,
# en ese orden. Las variables se prepararon dos celdas más arriba.
# No uses datos de test para el ajuste.



In [ ]:
# TODO: predice test una sola vez y guarda las etiquetas en predicciones_test.
# Pista: predicciones_test = estimador_final.predict(...)
# Completa el argumento con las características de test, no con sus etiquetas.
# Puedes comprobar que len(predicciones_test) coincide con len(y_dig_test).
# Conserva estas predicciones para las métricas y la matriz; no uses test para cambiar el modelo.



In [ ]:
# TODO: calcula las cuatro métricas de test y guárdalas en metricas_test.
# Pista: metricas_test = metricas_multiclase(..., ...)
# Completa los argumentos con las etiquetas reales de test y las predicciones,
# en ese orden. La función está en la primera celda de código de la sección 1.
# Usa display(...) para mostrar la serie y cita sus valores en la respuesta 7.1.



In [ ]:
# TODO: crea la lista ordenada de clases de 0 a 9 y guárdala en clases_test.
# Pista: clases_test = sorted(SERIE_ETIQUETAS.unique())
# Sustituye SERIE_ETIQUETAS por la variable de etiquetas de dígitos definida en 2.0.
# Puedes mostrar la lista y comprobar que contiene las diez cifras.



In [ ]:
# TODO: calcula la matriz de confusión de test y guárdala en matriz_test.
# Pista: matriz_test = matriz_confusion_multiclase(..., ..., clases_test)
# Completa los dos primeros argumentos con las etiquetas reales y predichas de test,
# en ese orden. La función está en la segunda celda de código de la sección 1.
# Utiliza las predicciones ya guardadas; no vuelvas a predecir.



In [ ]:
# TODO: muestra la matriz de test en una tabla con cifras como índice y columnas.
# Pista: tabla_test = pd.DataFrame(..., index=clases_test, columns=...)
# Completa con la matriz calculada y la lista de clases; después usa display(tabla_test).
# Puedes añadir .rename_axis(index="Predicción", columns="Real") para nombrar los ejes.
# Comprueba que la suma de los recuentos coincide con el número de ejemplos de test.



In [ ]:
# TODO: representa gráficamente la matriz de confusión de test.
# Pista: plt.imshow(..., cmap="Blues") dibuja la matriz; completa con la variable calculada.
# Añade plt.colorbar() para interpretar los colores y un título con plt.title(...).
# Usa plt.xticks(..., clases_test) y plt.yticks(..., clases_test) para marcar las cifras;
# las posiciones pueden obtenerse con range(len(clases_test)).
# Etiqueta el eje horizontal como clase real y el vertical como predicción con
# plt.xlabel(...) y plt.ylabel(...). Termina con plt.tight_layout() y plt.show().
# Compara el gráfico con la tabla y describe las confusiones en la respuesta 7.1.



<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 7.1</strong><br><br><b>1.</b> ¿Qué valores obtuviste para exactitud, F1 macro, precisión macro y sensibilidad macro?<br><br><b>2.</b> ¿Qué cifras se confunden más en la matriz y qué limitación tiene esta evaluación?<br><br><b>3.</b> ¿Por qué se reserva el conjunto de test hasta después de elegir el modelo y por qué no debe utilizarse para modificarlo?<br><br><em>Pista: lee la matriz con predicción en filas y realidad en columnas. El test no garantiza el mismo rendimiento en otros dígitos o contextos.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br><br>Tu respuesta aquí. Cita las cuatro métricas del test, describe una confusión de la matriz y explica por qué el test no se usa para escoger el modelo.</div>

## 8. Revisión final de la práctica

Antes de entregar, reinicia el kernel y ejecuta el notebook en orden y completa todas las respuestas. Las comprobaciones automáticas no pueden juzgar si las interpretaciones son adecuadas ni si consultaste test durante la selección.

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 8.1 — Clustering</strong><br><br><b>1.</b> ¿Qué papel tienen las etiquetas externas al formar grupos y al comprobar la partición elegida?<br><br><b>2.</b> ¿Qué evidencia usaste para elegir el método de clustering y el número de grupos? Indica también una limitación de tu análisis.<br><br><em>Pista: cita criterios o resultados internos que apoyen tu decisión y recuerda cuándo calculaste el ARI.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br><br>Tu respuesta aquí. Explica el papel de las etiquetas externas, resume la evidencia que apoyó tu elección del método y del número de grupos y menciona una limitación.</div>

<div style="background-color:#eff6ff; border-left:5px solid #2563eb; padding:10px; color:#1e3a8a;"><strong>PREGUNTA 8.2 — Clasificación</strong><br><br><b>1.</b> ¿Qué papel tienen las etiquetas al entrenar y evaluar el clasificador?<br><br><b>2.</b> ¿Qué evidencia usaste para elegir el clasificador? Indica también una limitación de tu evaluación.<br><br><em>Pista: cita resultados de validación que apoyen tu decisión y distingue su papel del test reservado para la evaluación final.</em></div>

<div style="background-color:#ecfdf5; border-left:5px solid #16a34a; padding:10px; color:#14532d;"><strong>RESPUESTA</strong><br><br>Tu respuesta aquí. Explica cómo se usan las etiquetas para entrenar y evaluar, resume la evidencia que apoyó tu elección del clasificador y menciona una limitación.</div>

### Comprobación automática

Después de completar la práctica, ejecuta la celda siguiente. Comprueba que existen la partición de clustering y la selección, y que las predicciones y métricas finales tienen los elementos esperados. No valida los gráficos, la interpretación escrita ni que hayas respetado el protocolo de selección.

In [ ]:
# Comprueba algunos resultados y tamaños al terminar la práctica.
verificaciones = {
    "Una etiqueta de clustering por observación": (
        "clusters_grupos" in globals()
        and len(clusters_grupos) == len(X_sintetico)
    ),
    "Método y k elegidos válidos": (
        "nombre_metodo" in globals()
        and nombre_metodo in {"K-Means", "Ward"}
        and "k_elegido" in globals()
        and k_elegido in valores_k
    ),
    "Clasificador elegido entre los candidatos": (
        "modelo_clasificador_elegido" in globals()
        and modelo_clasificador_elegido in candidatos
    ),
    "Una predicción de test por observación": (
        "predicciones_test" in globals()
        and len(predicciones_test) == len(y_dig_test)
    ),
    "Cuatro métricas de test disponibles": (
        "metricas_test" in globals()
        and isinstance(metricas_test, pd.Series)
        and {"Exactitud", "F1 macro", "Precisión macro", "Sensibilidad macro"}.issubset(metricas_test.index)
    ),
}

display(pd.DataFrame({"Correcto": pd.Series(verificaciones)}))
if all(verificaciones.values()):
    print("Comprobaciones superadas. Revisa también las respuestas y el protocolo de evaluación.")
else:
    print("Hay comprobaciones pendientes. Revisa las celdas anteriores y vuelve a ejecutar esta celda.")